# C9-dimensionality-reduction — Practice p24

**Type:** challenge · **Difficulty:** advanced · **Concepts:** numpy-pca-class-from-scratch, pca-black-box-insufficiency

Implement the complete p23 `NumpyPCA` contract and the function
`subspace_projector(model)`.
The function accepts a fitted `NumpyPCA` and returns
`model.components_.T @ model.components_` as a finite float square array.
It must reject an unfitted model with `ValueError`.

This challenge grades two degenerate regimes.

1. A rank-two centered matrix in four features: fitting two components must reconstruct exactly up to `ATOL`, and a four-component fit must report two trailing zero variances.
2. A matrix with a repeated largest covariance eigenvalue: any orthonormal basis of the top-two eigenspace is valid, so the checker compares only subspace projectors.

Do not sign-fix, sort, or compare repeated component rows against a hidden row order.
The reusable class still uses sample covariance with denominator `n - 1`, exactly one `np.linalg.eigh` call per successful fit, full-spectrum explained ratios, and the validation/data-flow contract from p22–p23.

**Zero points:** sklearn PCA, scipy PCA, or helpers that invoke them.
Use `ATOL = 1e-10`, `RTOL = 0.0`; do not mutate inputs.


In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0


class NumpyPCA:
    def __init__(self, n_components):
        # YOUR CODE HERE
        ...

    def fit(self, X):
        # YOUR CODE HERE
        ...

    def transform(self, X):
        # YOUR CODE HERE
        ...

    def fit_transform(self, X):
        # YOUR CODE HERE
        ...

    def inverse_transform(self, Z):
        # YOUR CODE HERE
        ...


def subspace_projector(model):
    # YOUR CODE HERE
    ...

## Immutable contract check — do not edit

The repeated fixture deliberately permits different signs, row order, and rotations.
Only covariance spectra, projectors, transformations, and reconstructions are compared.


In [ ]:
import dis
import inspect
import types

_ORIGINAL_EIGH_P24 = np.linalg.eigh


def _audit_p24():
    pending = [subspace_projector]
    for value in vars(NumpyPCA).values():
        if isinstance(value, (staticmethod, classmethod)):
            value = value.__func__
        if isinstance(value, types.FunctionType):
            pending.append(value)
    seen = set()
    while pending:
        function = pending.pop()
        if id(function) in seen:
            continue
        seen.add(id(function))
        codes = [function.__code__]
        while codes:
            code = codes.pop()
            codes.extend(item for item in code.co_consts if isinstance(item, types.CodeType))
            assert not ({"sklearn", "scipy"} & {name.lower() for name in code.co_names})
            for instruction in dis.get_instructions(code):
                if instruction.opname in {"IMPORT_NAME", "IMPORT_FROM"}:
                    assert not str(instruction.argval).lower().startswith(("sklearn", "scipy"))
            for name in code.co_names:
                value = function.__globals__.get(name)
                if isinstance(value, types.FunctionType):
                    pending.append(value)
                assert not str(getattr(value, "__module__", "")).lower().startswith(("sklearn", "scipy"))
    try:
        source = inspect.getsource(NumpyPCA).lower() + inspect.getsource(subspace_projector).lower()
    except (OSError, TypeError):
        source = ""
    assert "sklearn" not in source and "scipy" not in source


_audit_p24()

try:
    subspace_projector(NumpyPCA(1))
except ValueError:
    pass
else:
    raise AssertionError("unfitted projector request must raise ValueError")

_latent_p24 = np.array([
    [-3.0, 1.0], [-1.0, -2.0], [0.0, 1.0],
    [1.0, 3.0], [2.0, -1.0], [1.0, -2.0],
])
_mix_p24 = np.array([[1.0, 2.0, -1.0, 0.5], [0.0, 1.0, 2.0, -1.0]])
_X_rank_p24 = _latent_p24 @ _mix_p24 + np.array([7.0, -3.0, 2.0, 5.0])
assert np.linalg.matrix_rank(_X_rank_p24 - _X_rank_p24.mean(axis=0)) == 2
_X_rank_before_p24 = _X_rank_p24.copy()
_eigh_calls_p24 = []
def _traced_eigh_p24(matrix):
    _eigh_calls_p24.append(np.array(matrix, copy=True))
    return _ORIGINAL_EIGH_P24(matrix)
np.linalg.eigh = _traced_eigh_p24
_rank_model_p24 = NumpyPCA(2)
_Z_rank_p24 = _rank_model_p24.fit_transform(_X_rank_p24)
_Xhat_rank_p24 = _rank_model_p24.inverse_transform(_Z_rank_p24)
_P_rank_p24 = subspace_projector(_rank_model_p24)
assert np.array_equal(_X_rank_p24, _X_rank_before_p24)
assert isinstance(_P_rank_p24, np.ndarray) and _P_rank_p24.shape == (4, 4)
assert np.issubdtype(_P_rank_p24.dtype, np.floating) and np.isfinite(_P_rank_p24).all()
assert np.allclose(_P_rank_p24, _P_rank_p24.T, atol=ATOL, rtol=RTOL)
assert np.allclose(_P_rank_p24 @ _P_rank_p24, _P_rank_p24, atol=ATOL, rtol=RTOL)
assert np.isclose(np.trace(_P_rank_p24), 2.0, atol=ATOL, rtol=RTOL)
_C_rank_p24 = (_X_rank_p24 - _X_rank_p24.mean(axis=0)).T @ (_X_rank_p24 - _X_rank_p24.mean(axis=0)) / 5
_evals_rank_p24, _evecs_rank_p24 = _ORIGINAL_EIGH_P24(_C_rank_p24)
_Q_rank_ref_p24 = _evecs_rank_p24[:, np.argsort(_evals_rank_p24)[::-1][:2]].T
_P_rank_ref_p24 = _Q_rank_ref_p24.T @ _Q_rank_ref_p24
assert np.allclose(_P_rank_p24, _P_rank_ref_p24, atol=ATOL, rtol=RTOL)
assert np.allclose(_Xhat_rank_p24, _X_rank_p24, atol=ATOL, rtol=RTOL)

_rank_full_p24 = NumpyPCA(4).fit(_X_rank_p24)
assert np.allclose(_rank_full_p24.explained_variance_[2:], 0.0, atol=ATOL, rtol=RTOL)
assert np.isclose(_rank_full_p24.explained_variance_ratio_.sum(), 1.0, atol=ATOL, rtol=RTOL)

_X_repeat_p24 = np.array([
    [np.sqrt(2.0), 0.0, 0.0, 0.0],
    [-np.sqrt(2.0), 0.0, 0.0, 0.0],
    [0.0, np.sqrt(2.0), 0.0, 0.0],
    [0.0, -np.sqrt(2.0), 0.0, 0.0],
])
_repeat_model_p24 = NumpyPCA(2).fit(_X_repeat_p24)
np.linalg.eigh = _ORIGINAL_EIGH_P24
assert len(_eigh_calls_p24) == 3
_P_repeat_p24 = subspace_projector(_repeat_model_p24)
_P_repeat_ref_p24 = np.diag([1.0, 1.0, 0.0, 0.0])
assert np.allclose(_repeat_model_p24.explained_variance_[0], _repeat_model_p24.explained_variance_[1], atol=ATOL, rtol=RTOL)
assert np.allclose(_P_repeat_p24, _P_repeat_ref_p24, atol=ATOL, rtol=RTOL)
assert np.allclose(_P_repeat_p24 @ _P_repeat_p24, _P_repeat_p24, atol=ATOL, rtol=RTOL)
_repeat_hat_p24 = _repeat_model_p24.inverse_transform(_repeat_model_p24.transform(_X_repeat_p24))
assert np.allclose(_repeat_hat_p24, _X_repeat_p24, atol=ATOL, rtol=RTOL)

_theta_p24 = 0.41
_R_p24 = np.array([[np.cos(_theta_p24), -np.sin(_theta_p24)], [np.sin(_theta_p24), np.cos(_theta_p24)]])
_rotated_basis_p24 = _R_p24 @ _repeat_model_p24.components_
assert not np.allclose(_rotated_basis_p24, _repeat_model_p24.components_, atol=ATOL, rtol=RTOL)
assert np.allclose(_rotated_basis_p24.T @ _rotated_basis_p24, _P_repeat_p24, atol=ATOL, rtol=RTOL)